In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import os
import random

def fijar_semillas(semilla=42):
    # 1. Fijar semilla de Python
    os.environ['PYTHONHASHSEED'] = str(semilla)
    random.seed(semilla)
    
    # 2. Fijar semilla de NumPy
    np.random.seed(semilla)
    
    # 3. Fijar semilla de TensorFlow/Keras
    tf.random.set_seed(semilla)
    
    print(f"[*] Semillas fijadas a {semilla} para asegurar reproducibilidad.")

# Llamar a la función antes de crear ningún modelo ni dividir datos
fijar_semillas(42)

[*] Semillas fijadas a 42 para asegurar reproducibilidad.


In [3]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS (Código del Profesor)
# =====================================================================
print("Descargando datos de Yahoo Finance...")
start_date = '1960-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()
print(f"Forma de los datos de retornos: {returns.shape}")

# Función del profesor para crear ventanas
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    num_features = data_array.shape[1] 

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)
        
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) 
            y.append(average_output)
        else:
            y.append(data_array[i + input_window_size - 1])
            
    return np.array(X), np.array(y)

Descargando datos de Yahoo Finance...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Forma de los datos de retornos: (16195, 23)


In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

def construir_modelo_mixto(config, input_shape, n_assets=23):
    """Construye un modelo Híbrido: CNN -> LSTM -> Densa"""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # ---------------------------------------------------------
    # BLOQUE 1: EXTRACCIÓN DE PATRONES (CNN)
    # ---------------------------------------------------------
    # Usamos padding='same' para no reducir drásticamente la longitud en ventanas In:5
    model.add(Conv1D(filters=config['filtros_cnn'], 
                     kernel_size=config['kernel_size'], 
                     padding='same', 
                     activation='relu'))
    
    # Solo aplicamos MaxPooling si la ventana es de 10 días o más para no quedarnos sin datos
    if input_shape[0] >= 10:
        model.add(MaxPooling1D(pool_size=2))
        
    # ---------------------------------------------------------
    # BLOQUE 2: MEMORIA TEMPORAL (RNN)
    # ---------------------------------------------------------
    model.add(config['tipo_rnn'](config['neuronas_rnn'], return_sequences=False))
    
    # ---------------------------------------------------------
    # BLOQUE 3: TOMA DE DECISIONES (Densas)
    # ---------------------------------------------------------
    model.add(Dropout(config['dropout']))
    
    # Capa densa intermedia para procesar la salida de la LSTM
    if config['neuronas_densa'] > 0:
        model.add(Dense(config['neuronas_densa'], activation='relu'))
        
    # Salida: Regresión (23 activos)
    model.add(Dense(n_assets)) 
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='mae')
    
    return model


def calcular_baselines(X_test, y_test, y_train_mean):
    """Calcula el MAE para modelos simples, incluyendo Buy and Hold."""
    
    # 1. Baseline Naive: El futuro será igual al último día de la ventana de entrada
    y_pred_naive = X_test[:, -1, :]
    mae_naive = np.mean(np.abs(y_pred_naive - y_test))
    
    # 2. Baseline SMA: El futuro será igual a la media de la ventana de entrada actual
    y_pred_sma = np.mean(X_test, axis=1)
    mae_sma = np.mean(np.abs(y_pred_sma - y_test))
    
    # 3. Baseline Buy and Hold: Predecir siempre la media histórica del entrenamiento
    # Creamos un array del mismo tamaño que y_test relleno con la media de y_train
    y_pred_bh = np.full_like(y_test, y_train_mean)
    mae_bh = np.mean(np.abs(y_pred_bh - y_test))
    
    return mae_naive, mae_sma, mae_bh

# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia', exist_ok=True)

In [5]:
# =====================================================================
# BANCOS DE PRUEBAS PARA REDES MIXTAS (CNN + RNN + Dense)
# =====================================================================
input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

# Para ventanas MUY CORTAS (In: 5, 10)
# Queremos un Kernel pequeño (2 o 3) porque hay pocos días para buscar patrones.
hp_mixto_corto = [
    # Modelo Ágil: Pocos filtros, memoria ligera.
    {'filtros_cnn': 16, 'kernel_size': 2, 'tipo_rnn': GRU,  'neuronas_rnn': 16, 'neuronas_densa': 0,  'dropout': 0.1, 'lr': 0.001},
    
    # Modelo Balanceado: Kernel de 3 días (patrones de media semana).
    {'filtros_cnn': 32, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 32, 'neuronas_densa': 16, 'dropout': 0.2, 'lr': 0.0005},
    
    # Modelo Profundo: Busca patrones complejos y los remata con una densa fuerte.
    {'filtros_cnn': 64, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 32, 'neuronas_densa': 32, 'dropout': 0.2, 'lr': 0.0005}
]

# Para ventanas LARGAS (In: 30, 90)
# Podemos usar Kernels más grandes (5 días = una semana bursátil entera de patrón).
hp_mixto_largo = [
    # Escáner Semanal: Kernel 5 busca patrones semanales dentro del mes/trimestre.
    {'filtros_cnn': 32, 'kernel_size': 5, 'tipo_rnn': GRU,  'neuronas_rnn': 32, 'neuronas_densa': 16, 'dropout': 0.2, 'lr': 0.0005},
    
    # Escáner Rápido y Memoria Fuerte: Patrones de 3 días pero mucha memoria LSTM.
    {'filtros_cnn': 64, 'kernel_size': 3, 'tipo_rnn': LSTM, 'neuronas_rnn': 64, 'neuronas_densa': 32, 'dropout': 0.3, 'lr': 0.0005},
    
    # El "Monstruo" Mixto (Riesgo de Overfitting, pero hay que probarlo).
    {'filtros_cnn': 128, 'kernel_size': 5, 'tipo_rnn': LSTM, 'neuronas_rnn': 64, 'neuronas_densa': 32, 'dropout': 0.3, 'lr': 0.0001}
]

In [ ]:

# Matrices para reportar resultados finales de las Redes Recurrentes
matriz_mae_train_rnn = np.zeros((4, 4))
matriz_mae_val_rnn = np.zeros((4, 4))
matriz_mae_rnn = np.zeros((4, 4)) # Esta es la de Test que ya tenías

matriz_mae_naive = np.zeros((4, 4))  #test
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

# Aumentamos la paciencia a 15 épocas
early_stop = EarlyStopping(
    monitor ='val_loss', 
    patience = 5,               # <--- CAMBIO AQUÍ
    restore_best_weights = True  # IMPORTANTE: Que devuelva los pesos de la mejor época
)

In [10]:
# =====================================================================
# 4. BUCLE PRINCIPAL (AUTOMATIZACIÓN DE LOS 16 MODELOS x CONFIGURACIONES)
# =====================================================================


# =====================================================================
# RECORDATORIO: Inicializa estas nuevas matrices antes del bucle
# =====================================================================
matriz_mae_naive_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_sma_val = np.zeros((len(input_windows), len(output_windows)))
matriz_mae_bh_val = np.zeros((len(input_windows), len(output_windows)))

print("\nIniciando entrenamiento de modelos...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        print(f"\n=======================================================")
        print(f" Ventana Entrada: {in_w} días | Ventana Salida: {out_w} días")
        print(f"=======================================================")
        
        # 1. Crear datos
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # 2. Separación CRONOLÓGICA: 80% Train, 10% Validacion, 10% Test
        # split_1 = int(len(X) * 0.8)
        # split_2 = int(len(X) * 0.9)
        
        # Para un esquema 70% Train, 20% Validacion, 10% Test
        split_1 = int(len(X) * 0.70) # Aquí cortamos el Train
        split_2 = int(len(X) * 0.90) # Aquí cortamos la Validación (del 70% al 90% = 20%)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]

        # Calculamos la media global de entrenamiento para esta ventana (Buy and Hold)
        y_train_mean = np.mean(y_train, axis=0)
        
        
        # =====================================================================
        # 3. Baselines (AHORA EN VALIDACIÓN Y TEST)
        # =====================================================================

        '''
        mae_naive, mae_sma, mae_bh = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive
        matriz_mae_sma[i, j] = mae_sma
        matriz_mae_bh[i, j] = mae_bh

        print(f"Baseline Naive (MAE en Test): {mae_naive:.6f}")
        print(f"Baseline SMA   (MAE en Test): {mae_sma:.6f}")
        print(f"Baseline Buy & Hold (MAE): {mae_bh:.6f}")
        '''

        # Calcular en Validación
        mae_naive_val, mae_sma_val, mae_bh_val = calcular_baselines(X_val, y_val, y_train_mean)
        matriz_mae_naive_val[i, j] = mae_naive_val
        matriz_mae_sma_val[i, j] = mae_sma_val
        matriz_mae_bh_val[i, j] = mae_bh_val
        
        # Calcular en Test
        mae_naive_test, mae_sma_test, mae_bh_test = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive_test
        matriz_mae_sma[i, j] = mae_sma_test
        matriz_mae_bh[i, j] = mae_bh_test
        

        print("--- Baselines VALIDACIÓN ---")
        print(f"Naive: {mae_naive_val:.6f} | SMA: {mae_sma_val:.6f} | Buy&Hold: {mae_bh_val:.6f}")
        print("--- Baselines TEST ---")
        print(f"Naive: {mae_naive_test:.6f} | SMA: {mae_sma_test:.6f} | Buy&Hold: {mae_bh_test:.6f}\n")


        # 4. Búsqueda del mejor modelo recurrente
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None


        # DIVIDE Y VENCERÁS: Selección de hiperparámetros

        if in_w in [5, 10]:
            lista_a_probar = hp_mixto_corto
            nombre_lista = "Mixto Corto"
        else:
            lista_a_probar = hp_mixto_largo
            nombre_lista = "Mixto Largo"

        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        for config in lista_a_probar:
            nombre_rnn = config['tipo_rnn'].__name__
            print(f" -> Entrenando Mixto: CNN({config['filtros_cnn']}F, K{config['kernel_size']}) + {nombre_rnn}({config['neuronas_rnn']}) + Densa({config['neuronas_densa']})")
            
            # Llamamos a TU función de arquitectura mixta
            modelo = construir_modelo_mixto(config, input_shape=(in_w, 23))
            
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop], 
                                   verbose=0)
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n -> Ganador Mixto: CNN({mejor_config['filtros_cnn']}F, K{mejor_config['kernel_size']}) + {nombre_rnn}({mejor_config['neuronas_rnn']}) + Densa({mejor_config['neuronas_densa']})")
        
        # 5. Evaluación final del GANADOR en TRAIN, VALIDACIÓN y TEST
        mae_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        mae_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        mae_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        
        # Guardar en sus respectivas matrices (¡recuerda inicializarlas antes del bucle!)
        matriz_mae_train_rnn[i, j] = mae_train_ganador
        matriz_mae_val_rnn[i, j] = mae_val_ganador
        matriz_mae_rnn[i, j] = mae_test_ganador
        
        print(f"MAE del Modelo Ganador en TRAIN:      {mae_train_ganador:.6f}")
        print(f"MAE del Modelo Ganador en VALIDACIÓN: {mae_val_ganador:.6f}")
        print(f"MAE del Modelo Ganador en TEST:       {mae_test_ganador:.6f}")
        
        # 6. Guardar Gráfica de Convergencia del Ganador
        plt.figure(figsize=(12, 6)) # Hacemos la gráfica un poco más ancha para el título largo
        plt.plot(mejor_historial.history['loss'], label='Error Entrenamiento (MAE)')
        plt.plot(mejor_historial.history['val_loss'], label='Error Validación (MAE)')
        
        # Extraer los hiperparámetros de la arquitectura Mixta
        f_cnn = mejor_config['filtros_cnn']
        k_cnn = mejor_config['kernel_size']
        nombre_rnn = mejor_config['tipo_rnn'].__name__
        n_rnn = mejor_config['neuronas_rnn']
        n_densa = mejor_config['neuronas_densa']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        # Construir un título que resuma la arquitectura completa
        titulo_arqui = f"CNN({f_cnn}F, K{k_cnn}) + {nombre_rnn}({n_rnn}) + Densa({n_densa})"
        plt.title(f"Convergencia Mixta: {titulo_arqui}\nLR: {l_rate} | Drop: {d_out} | (Ventana In:{in_w} - Out:{out_w})")
        
        plt.xlabel('Épocas')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(True)
        
        # Guardar la imagen en una nueva carpeta para no pisar las viejas
        os.makedirs('graficas_convergencia_mixtas', exist_ok=True)
        nombre_archivo = f"graficas_convergencia_mixtas/conver_mixto_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos...

 Ventana Entrada: 5 días | Ventana Salida: 1 días
--- Baselines VALIDACIÓN ---
Naive: 0.015404 | SMA: 0.011784 | Buy&Hold: 0.010569
--- Baselines TEST ---
Naive: 0.017819 | SMA: 0.013634 | Buy&Hold: 0.012266

 -> Usando banco de pruebas: [Mixto Corto]
 -> Entrenando Mixto: CNN(16F, K2) + GRU(16) + Densa(0)
 -> Entrenando Mixto: CNN(32F, K3) + LSTM(32) + Densa(16)
 -> Entrenando Mixto: CNN(64F, K3) + LSTM(32) + Densa(32)

 -> Ganador Mixto: CNN(64F, K3) + LSTM(32) + Densa(32)
MAE del Modelo Ganador en TRAIN:      0.011830
MAE del Modelo Ganador en VALIDACIÓN: 0.010575
MAE del Modelo Ganador en TEST:       0.012271

 Ventana Entrada: 5 días | Ventana Salida: 5 días
--- Baselines VALIDACIÓN ---
Naive: 0.011804 | SMA: 0.006918 | Buy&Hold: 0.004729
--- Baselines TEST ---
Naive: 0.013680 | SMA: 0.008045 | Buy&Hold: 0.005591

 -> Usando banco de pruebas: [Mixto Corto]
 -> Entrenando Mixto: CNN(16F, K2) + GRU(16) + Densa(0)
 -> Entrenando Mixto: CNN(32F,

In [ ]:
# =====================================================================
# 5. RESULTADOS FINALES (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (RNN)")
print("="*50)
df_rnn_train = pd.DataFrame(matriz_mae_train_rnn, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_train)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (RNN)")
print("="*50)
df_rnn_val = pd.DataFrame(matriz_mae_val_rnn, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS FINALES EN TEST (RNN)")
print("="*50)
df_rnn = pd.DataFrame(matriz_mae_rnn, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_rnn)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (TEST)")
print("="*50)
df_naive = pd.DataFrame(matriz_mae_naive, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE NAIVE (VALIDACION)")
print("="*50)
df_naive_val = pd.DataFrame(matriz_mae_naive_val, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (TEST)")
print("="*50)
df_sma = pd.DataFrame(matriz_mae_sma, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE SMA (VALIDACION)")
print("="*50)
df_sma_val = pd.DataFrame(matriz_mae_sma_val, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma_val)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh = pd.DataFrame(matriz_mae_bh, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh)

print("\n" + "="*50)
print("MATRIZ DE RESULTADOS BASELINE BUY AND HOLD (TEST)")
print("="*50)
df_bh_val = pd.DataFrame(matriz_mae_bh_val, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh_val)